<a href="https://colab.research.google.com/github/rathorebharat/mftracker/blob/main/mf-correction-engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# SECTION 0 — IMPORTS / DEPENDENCIES
# ============================================================

import requests
import time
import warnings

import numpy as np
import pandas as pd

from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

In [10]:
# ============================================================
# SECTION 1 — CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# TIGZIG NAV API
# ------------------------------------------------------------
#
# IMPORTANT:
# Keep the same TigZig URL that you were already using.
#
# Example:
# TIGZIG_NAV_URL = "YOUR_EXISTING_TIGZIG_NAV_ENDPOINT"
#
# Do NOT substitute mftool here.
# ------------------------------------------------------------

TIGZIG_NAV_URL = "YOUR_EXISTING_TIGZIG_NAV_URL"


# ------------------------------------------------------------
# FUND CODES
# ------------------------------------------------------------

codes = [
    119277, 120620, 143341, 119063, 141877,
    133516, 148703, 152430, 152417, 120492,
    142110, 141226, 120351, 120465, 118834,
    120152, 120505, 122639, 145454, 143783,
    147409, 120594, 120578, 120728, 119705,
    145552, 127042, 138528, 140243, 103819,
    119218, 119727, 119151, 150597, 148623,
    112932, 120733, 135781, 120334, 118825,
    153196, 125354, 109445, 120244, 119544,
    118989, 119775, 130503, 119556, 119069,
    119714
]


# ------------------------------------------------------------
# FUND NAME MAP
#
# This is deliberately kept separately from the NAV engine.
# If TigZig returns a fund name, that name will be used.
# Otherwise this map supplies it.
# ------------------------------------------------------------

FUND_NAMES = {

    109445: "ICICI Prudential Banking & Financial Services",
    120244: "ICICI Prudential Banking & Financial Services Direct",
    120733: "UTI Banking & Financial Services",
    148623: "Mirae Asset Banking & Financial Services",

    119063: "HDFC Nifty 50",
    120620: "ICICI Prudential Nifty 50",
    143341: "UTI Nifty Next 50",

    120152: "Kotak Large Cap",
    120465: "Axis Large Cap",
    118825: "Mirae Asset Large Cap",

    103819: "DSP Large & Mid Cap - Regular",
    119218: "DSP Large & Mid Cap - Direct",
    112932: "Mirae Asset Large & Midcap - Regular",
    118834: "Mirae Asset Large & Midcap - Direct",

    120492: "JM Flexicap",
    122639: "Parag Parikh Flexi Cap",
    141226: "Mahindra Manulife Multi Cap",

    120505: "Axis Midcap",
    142110: "Mahindra Manulife Mid Cap",
    119775: "Kotak Midcap",

    118989: "HDFC Mid Cap",
    127042: "Motilal Oswal Midcap",

    125354: "Axis Small Cap",
    119556: "Aditya Birla Sun Life Small Cap",
    153196: "Mirae Asset Small Cap",
    130503: "HDFC Small Cap",

    152430: "HDFC Nifty 200 Momentum 30",
    148703: "UTI Nifty 200 Momentum 30",

    120594: "ICICI Prudential Technology",
    120578: "SBI Technology Opportunities",
    150597: "Mirae Asset Global X AI & Technology FoF",

    143783: "Mirae Asset Healthcare",
    147409: "ABSL Pharma & Healthcare",
    145454: "DSP Healthcare",

    120728: "UTI Infrastructure",
    120351: "LIC MF Infrastructure",
    133516: "ABSL Manufacturing",
    119705: "SBI COMMA",

    138528: "PGIM India Global Equity Opportunities FoF",
    140243: "Edelweiss Greater China",
    145552: "Motilal Oswal Nasdaq 100 FoF",
    119277: "DSP World Gold Mining FoF",

    135781: "Mirae Asset ELSS",
    119544: "Aditya Birla Sun Life ELSS",

    119727: "SBI Focused Fund",

    120334: "ICICI Prudential Multi-Asset",

    119714: "SBI Medium to Long Duration Fund-DIRECT PLAN -Growth",

    119151: "ESSEL FLEXIBLE INCOME FUND-DIRECT PLAN-GROWTH OPTION",

    119069: "HDFC Income Fund - Growth Option - Direct Plan",

}


# ------------------------------------------------------------
# STRATEGIC GROUPS
# ------------------------------------------------------------

strategic_groups = {

    "CORE_EQUITY": [
        119063, 120620,
        143341,
        120152, 120465, 118825,
        103819, 119218,
        112932, 118834,
    ],

    "GROWTH_EQUITY": [
        120492, 122639, 141226,
        120505, 142110, 119775,
        118989, 127042,
        125354, 119556, 153196, 130503,
    ],

    "FACTOR_EQUITY": [
        152430, 148703,
    ],

    "SECTOR_EQUITY": [
        109445, 120244,
        120733, 148623,

        120594, 120578, 150597,

        143783, 147409, 145454,

        120728, 120351,
        133516, 119705,
    ],

    "INTERNATIONAL": [
        138528,
        140243,
        145552,
        119277,
    ],

    "HYBRID_DEBT_OTHER": [
        135781,
        119544,
        119727,
        120334,
        119714,
    ],
}


# ------------------------------------------------------------
# INVESTED FUNDS
# ------------------------------------------------------------
#
# These are the funds already held in the portfolio.
# Keep this list aligned with your current portfolio.
# ------------------------------------------------------------

invested_codes = set([
    119063,
    141877,
    152430,
    122639,
    147409,
    120594,
    120578,
    119705,
    127042,
    138528,
    119218,
    119727,
    119714,
    148623,
    120733,
    118834,
    135781,
    120334,
    118825,
    153198,
    125354,
    120244,
    119544,
    118989,
    130503,
])


# ------------------------------------------------------------
# CORRECTION ENGINE PARAMETERS
# ------------------------------------------------------------

MIN_CORRECTION_PCT = 3.0

# Recovery to this percentage of the original peak
# is considered a completed correction episode.
RECOVERY_THRESHOLD = 0.995

# Small tolerance when comparing today's correction with
# historical corrections.
NEAR_TODAY_TOLERANCE = 1.0


# ------------------------------------------------------------
# PROFIT / OVERSHOOT PARAMETERS
# ------------------------------------------------------------

OVERSHOOT_PCT_1 = 10.0
OVERSHOOT_PCT_2 = 20.0
OVERSHOOT_PCT_3 = 30.0


# ------------------------------------------------------------
# INVESTMENT PARAMETERS
# ------------------------------------------------------------

MONTHLY_INVESTMENT = 200000


# ------------------------------------------------------------
# ASSET CLASS THRESHOLDS
#
# These are deliberately kept simple.
# The historical episode percentiles remain the main
# correction evidence.
# ------------------------------------------------------------

ASSET_CLASS_MAP = {

    "CORE_EQUITY": "EQUITY",
    "GROWTH_EQUITY": "EQUITY",
    "FACTOR_EQUITY": "EQUITY",
    "SECTOR_EQUITY": "EQUITY",
    "INTERNATIONAL": "EQUITY",
    "HYBRID_DEBT_OTHER": "HYBRID_DEBT",

}

In [11]:
# ============================================================
# SECTION 2 — CONFIGURATION VALIDATION
# ============================================================

def validate_configuration():

    print("=" * 80)
    print("CONFIGURATION VALIDATION")
    print("=" * 80)

    # --------------------------------------------------------
    # Duplicate codes
    # --------------------------------------------------------

    duplicate_codes = [
        code
        for code in set(codes)
        if codes.count(code) > 1
    ]

    if duplicate_codes:
        print("WARNING — duplicate codes:", duplicate_codes)

    else:
        print("✓ No duplicate fund codes")


    # --------------------------------------------------------
    # Codes missing from name map
    # --------------------------------------------------------

    missing_names = [
        code
        for code in codes
        if code not in FUND_NAMES
    ]

    if missing_names:
        print(
            "WARNING — names missing for:",
            missing_names
        )

    else:
        print("✓ Every requested fund has a name")


    # --------------------------------------------------------
    # Group coverage
    # --------------------------------------------------------

    grouped_codes = set()

    for group, group_codes in strategic_groups.items():

        grouped_codes.update(group_codes)

    ungrouped = [
        code
        for code in codes
        if code not in grouped_codes
    ]

    if ungrouped:
        print(
            "WARNING — ungrouped funds:",
            ungrouped
        )

    else:
        print("✓ All requested funds belong to a strategic group")


    # --------------------------------------------------------
    # Group duplicates
    # --------------------------------------------------------

    group_occurrences = {}

    for group, group_codes in strategic_groups.items():

        for code in group_codes:

            group_occurrences.setdefault(
                code,
                []
            ).append(group)

    multi_group = {
        code: groups
        for code, groups
        in group_occurrences.items()
        if len(groups) > 1
    }

    if multi_group:

        print(
            "WARNING — funds belonging to multiple groups:"
        )

        for code, groups in multi_group.items():

            print(
                code,
                groups
            )

    else:
        print("✓ No strategic-group overlap")


    print()
    print(
        f"Requested funds : {len(codes)}"
    )

    print(
        f"Invested funds  : {len(invested_codes)}"
    )

    print()


validate_configuration()

CONFIGURATION VALIDATION
✓ No duplicate fund codes
WARNING — names missing for: [141877, 152417]
WARNING — ungrouped funds: [141877, 152417, 119151, 119069]
✓ No strategic-group overlap

Requested funds : 51
Invested funds  : 25



In [14]:
# ============================================================
# SECTION 3 — TIGZIG NAV FETCH ENGINE
# ============================================================


# TigZig API
TIGZIG_NAV_URL = (
    f"https://api.tigzig.com/mf/v1/nav"
)


# Historical analysis window
ANALYSIS_YEARS = 5

def get_tigzig_history(scheme_code):

    response = requests.get(
        TIGZIG_NAV_URL,
        params={"scheme": int(scheme_code)},
        timeout=60
    )

    response.raise_for_status()

    result = response.json()

    if "data" not in result:

        raise ValueError(
            f"No NAV data returned for scheme {scheme_code}"
        )

    data = result["data"]

    if not data:

        raise ValueError(
            f"Empty NAV dataset for scheme {scheme_code}"
        )

    df = pd.DataFrame(data)

    if df.empty:

        raise ValueError(
            f"Empty dataframe for scheme {scheme_code}"
        )

    return df


# ------------------------------------------------------------
# Normalize one TigZig response
# ------------------------------------------------------------

def normalize_tigzig_nav(
    raw_df,
    scheme_code
):

    df = raw_df.copy()

    # --------------------------------------------------------
    # Identify date column
    # --------------------------------------------------------

    date_candidates = [
        "Date",
        "date",
        "NAVDate",
        "nav_date"
    ]

    date_col = next(
        (
            c
            for c in date_candidates
            if c in df.columns
        ),
        None
    )

    if date_col is None:

        raise ValueError(
            f"Could not identify Date column for "
            f"scheme {scheme_code}. "
            f"Columns={list(df.columns)}"
        )


    # --------------------------------------------------------
    # Identify NAV column
    # --------------------------------------------------------

    nav_candidates = [
        "NAV",
        "nav",
        "Net Asset Value",
        "net_asset_value"
    ]

    nav_col = next(
        (
            c
            for c in nav_candidates
            if c in df.columns
        ),
        None
    )

    if nav_col is None:

        raise ValueError(
            f"Could not identify NAV column for "
            f"scheme {scheme_code}. "
            f"Columns={list(df.columns)}"
        )


    df = df.rename(
        columns={
            date_col: "Date",
            nav_col: "NAV"
        }
    )


    # --------------------------------------------------------
    # Standardize
    # --------------------------------------------------------

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    df["NAV"] = pd.to_numeric(
        df["NAV"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["Date", "NAV"]
    )

    df = df[df["NAV"] > 0].copy()

    df["Scheme_Code"] = int(scheme_code)

    # --------------------------------------------------------
    # Scheme name
    # --------------------------------------------------------

    if "Scheme_Name" not in df.columns:

        df["Scheme_Name"] = FUND_NAMES.get(
            int(scheme_code),
            "NoName"
        )

    else:

        df["Scheme_Name"] = (
            df["Scheme_Name"]
            .fillna(
                FUND_NAMES.get(
                    int(scheme_code),
                    "NoName"
                )
            )
        )


    # --------------------------------------------------------
    # Remove duplicate NAV dates
    # --------------------------------------------------------

    df = (
        df
        .sort_values("Date")
        .drop_duplicates(
            subset=["Scheme_Code", "Date"],
            keep="last"
        )
    )

    return df[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Date",
            "NAV"
        ]
    ]


# ------------------------------------------------------------
# Fetch ALL funds
# ------------------------------------------------------------

def fetch_all_nav_data(
    scheme_codes,
    pause_seconds=0.10
):

    frames = []
    not_found = []
    errors = []

    print("=" * 80)
    print("FETCHING HISTORICAL NAV FROM TIGZIG")
    print("=" * 80)

    for n, scheme_code in enumerate(
        scheme_codes,
        start=1
    ):

        print(
            f"[{n:02d}/{len(scheme_codes):02d}] "
            f"{scheme_code} "
            f"{FUND_NAMES.get(scheme_code, 'NoName')}"
        )

        try:

            raw = get_tigzig_history(
                scheme_code
            )

            normalized = normalize_tigzig_nav(
                raw,
                scheme_code
            )

            if normalized.empty:

                not_found.append(
                    scheme_code
                )

            else:

                frames.append(
                    normalized
                )

        except Exception as e:

            errors.append({
                "Scheme_Code": scheme_code,
                "Error": str(e)
            })

            print(
                f"    ERROR: {e}"
            )

        time.sleep(
            pause_seconds
        )


    # --------------------------------------------------------
    # IMPORTANT:
    # This ALWAYS creates df_nav.
    # --------------------------------------------------------

    if frames:

        df_nav = pd.concat(
            frames,
            ignore_index=True
        )

        df_nav = (
            df_nav
            .sort_values(
                [
                    "Scheme_Code",
                    "Date"
                ]
            )
            .reset_index(drop=True)
        )

    else:

        # Empty dataframe with correct structure
        df_nav = pd.DataFrame(
            columns=[
                "Scheme_Code",
                "Scheme_Name",
                "Date",
                "NAV"
            ]
        )


    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    print()
    print("=" * 80)
    print("NAV FETCH SUMMARY")
    print("=" * 80)

    print(
        f"Requested : {len(scheme_codes)}"
    )

    print(
        f"Returned  : "
        f"{df_nav['Scheme_Code'].nunique()}"
    )

    print(
        f"Not found : {not_found}"
    )

    print(
        f"Errors    : {len(errors)}"
    )

    print(
        f"Rows      : {len(df_nav):,}"
    )

    return (
        df_nav,
        pd.DataFrame(errors),
        not_found
    )

In [15]:
# ============================================================
# SECTION 4 — BUILD df_nav
# ============================================================

df_nav, fetch_errors_df, not_found = fetch_all_nav_data(
    codes
)

print()
print("df_nav shape:", df_nav.shape)

print()
print(df_nav.head())

print()
print(
    "Funds loaded:",
    df_nav["Scheme_Code"].nunique()
)

FETCHING HISTORICAL NAV FROM TIGZIG
[01/51] 119277 DSP World Gold Mining FoF
[02/51] 120620 ICICI Prudential Nifty 50
[03/51] 143341 UTI Nifty Next 50
[04/51] 119063 HDFC Nifty 50
[05/51] 141877 NoName
[06/51] 133516 ABSL Manufacturing
[07/51] 148703 UTI Nifty 200 Momentum 30
[08/51] 152430 HDFC Nifty 200 Momentum 30
[09/51] 152417 NoName
[10/51] 120492 JM Flexicap
[11/51] 142110 Mahindra Manulife Mid Cap
[12/51] 141226 Mahindra Manulife Multi Cap
[13/51] 120351 LIC MF Infrastructure
[14/51] 120465 Axis Large Cap
[15/51] 118834 Mirae Asset Large & Midcap - Direct
[16/51] 120152 Kotak Large Cap
[17/51] 120505 Axis Midcap
[18/51] 122639 Parag Parikh Flexi Cap
[19/51] 145454 DSP Healthcare
[20/51] 143783 Mirae Asset Healthcare
[21/51] 147409 ABSL Pharma & Healthcare
[22/51] 120594 ICICI Prudential Technology
[23/51] 120578 SBI Technology Opportunities
[24/51] 120728 UTI Infrastructure
[25/51] 119705 SBI COMMA
[26/51] 145552 Motilal Oswal Nasdaq 100 FoF
[27/51] 127042 Motilal Oswal Midcap


In [16]:
# ============================================================
# SECTION 5 — CORRECTION EPISODE ENGINE
# ============================================================

def build_correction_episodes(
    fund_df,
    min_correction=MIN_CORRECTION_PCT,
    recovery_threshold=RECOVERY_THRESHOLD
):

    df = (
        fund_df
        .sort_values("Date")
        .reset_index(drop=True)
        .copy()
    )

    if len(df) < 3:

        return pd.DataFrame()


    nav = df["NAV"].astype(float).values
    dates = df["Date"].values

    episodes = []

    peak_idx = 0
    trough_idx = 0

    in_correction = False

    for i in range(1, len(df)):

        current_nav = nav[i]


        # ====================================================
        # NO ACTIVE CORRECTION
        # ====================================================

        if not in_correction:

            if current_nav >= nav[peak_idx]:

                peak_idx = i
                trough_idx = i

                continue


            correction = (
                (nav[peak_idx] - current_nav)
                / nav[peak_idx]
                * 100
            )

            if correction >= min_correction:

                in_correction = True
                trough_idx = i

            continue


        # ====================================================
        # ACTIVE CORRECTION
        # ====================================================

        if current_nav < nav[trough_idx]:

            trough_idx = i

            continue


        peak_nav = nav[peak_idx]

        recovery_ratio = (
            current_nav / peak_nav
        )


        # ====================================================
        # CORRECTION COMPLETED
        # ====================================================

        if recovery_ratio >= recovery_threshold:

            trough_nav = nav[trough_idx]

            correction_pct = (
                (peak_nav - trough_nav)
                / peak_nav
                * 100
            )

            peak_date = pd.Timestamp(
                dates[peak_idx]
            )

            trough_date = pd.Timestamp(
                dates[trough_idx]
            )

            recovery_date = pd.Timestamp(
                dates[i]
            )

            episodes.append({

                "Peak_Date":
                    peak_date,

                "Peak_NAV":
                    peak_nav,

                "Trough_Date":
                    trough_date,

                "Trough_NAV":
                    trough_nav,

                "Recovery_Date":
                    recovery_date,

                "Recovery_NAV":
                    current_nav,

                "Correction_%":
                    correction_pct,

                "Peak_to_Trough_Days":
                    (
                        trough_date -
                        peak_date
                    ).days,

                "Trough_to_Recovery_Days":
                    (
                        recovery_date -
                        trough_date
                    ).days,

                "Total_Recovery_Days":
                    (
                        recovery_date -
                        peak_date
                    ).days,

                "Recovered_%":
                    100.0
            })


            # New cycle starts
            peak_idx = i
            trough_idx = i

            in_correction = False


    # ========================================================
    # OPEN CORRECTION
    # ========================================================

    if in_correction:

        peak_nav = nav[peak_idx]
        trough_nav = nav[trough_idx]
        current_nav = nav[-1]

        correction_pct = (
            (peak_nav - trough_nav)
            / peak_nav
            * 100
        )

        if correction_pct >= min_correction:

            recovered_pct = (
                (current_nav - trough_nav)
                /
                (peak_nav - trough_nav)
                * 100
            )

            episodes.append({

                "Peak_Date":
                    pd.Timestamp(
                        dates[peak_idx]
                    ),

                "Peak_NAV":
                    peak_nav,

                "Trough_Date":
                    pd.Timestamp(
                        dates[trough_idx]
                    ),

                "Trough_NAV":
                    trough_nav,

                "Recovery_Date":
                    pd.NaT,

                "Recovery_NAV":
                    np.nan,

                "Correction_%":
                    correction_pct,

                "Peak_to_Trough_Days":
                    (
                        pd.Timestamp(
                            dates[trough_idx]
                        )
                        -
                        pd.Timestamp(
                            dates[peak_idx]
                        )
                    ).days,

                "Trough_to_Recovery_Days":
                    np.nan,

                "Total_Recovery_Days":
                    np.nan,

                "Recovered_%":
                    recovered_pct
            })


    if not episodes:

        return pd.DataFrame()


    return pd.DataFrame(
        episodes
    )

In [17]:
# ============================================================
# SECTION 6 — BUILD ALL CORRECTION EPISODES
# ============================================================

def build_all_episodes(
    df_nav
):

    all_episodes = []

    for scheme_code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        episodes = build_correction_episodes(
            fund_df
        )

        if episodes.empty:
            continue

        episodes["Scheme_Code"] = scheme_code

        episodes["Scheme_Name"] = (
            fund_df["Scheme_Name"].iloc[0]
        )

        all_episodes.append(
            episodes
        )


    if all_episodes:

        episodes_df = pd.concat(
            all_episodes,
            ignore_index=True
        )

        episodes_df = episodes_df[
            [
                "Scheme_Code",
                "Scheme_Name",
                "Peak_Date",
                "Peak_NAV",
                "Trough_Date",
                "Trough_NAV",
                "Recovery_Date",
                "Recovery_NAV",
                "Correction_%",
                "Peak_to_Trough_Days",
                "Trough_to_Recovery_Days",
                "Total_Recovery_Days",
                "Recovered_%"
            ]
        ]

        episodes_df = (
            episodes_df
            .sort_values(
                [
                    "Scheme_Code",
                    "Peak_Date"
                ]
            )
            .reset_index(drop=True)
        )

    else:

        episodes_df = pd.DataFrame()


    return episodes_df


# ------------------------------------------------------------
# EXECUTE
# ------------------------------------------------------------

episodes_df = build_all_episodes(
    df_nav
)

print(
    "episodes_df shape:",
    episodes_df.shape
)

print(
    "Funds with episodes:",
    episodes_df["Scheme_Code"].nunique()
    if not episodes_df.empty
    else 0
)

episodes_df shape: (1414, 13)
Funds with episodes: 51


In [18]:
# ============================================================
# SECTION 7 — DIAGNOSTICS 1
# CURRENT CORRECTION / EPISODE STATE
# ============================================================

def get_current_state(
    fund_df,
    fund_episodes
):

    fund_df = (
        fund_df
        .sort_values("Date")
        .reset_index(drop=True)
    )

    current_date = fund_df["Date"].iloc[-1]
    current_nav = float(
        fund_df["NAV"].iloc[-1]
    )


    # --------------------------------------------------------
    # Historical completed episodes
    # --------------------------------------------------------

    if fund_episodes is not None:

        completed = fund_episodes[
            fund_episodes["Recovery_Date"].notna()
        ].copy()

        open_eps = fund_episodes[
            fund_episodes["Recovery_Date"].isna()
        ].copy()

    else:

        completed = pd.DataFrame()
        open_eps = pd.DataFrame()


    # --------------------------------------------------------
    # Current all-time peak
    # --------------------------------------------------------

    peak_idx = fund_df["NAV"].idxmax()

    peak_nav = float(
        fund_df.loc[
            peak_idx,
            "NAV"
        ]
    )

    peak_date = pd.Timestamp(
        fund_df.loc[
            peak_idx,
            "Date"
        ]
    )


    current_correction = (
        (peak_nav - current_nav)
        / peak_nav
        * 100
    )


    # --------------------------------------------------------
    # Determine current trough
    #
    # If current NAV is below peak, find lowest NAV
    # since that peak.
    # --------------------------------------------------------

    after_peak = fund_df[
        fund_df["Date"] >= peak_date
    ]

    if (
        current_correction > 0
        and len(after_peak) > 0
    ):

        trough_idx = after_peak["NAV"].idxmin()

        trough_nav = float(
            after_peak.loc[
                trough_idx,
                "NAV"
            ]
        )

        trough_date = pd.Timestamp(
            after_peak.loc[
                trough_idx,
                "Date"
            ]
        )

        total_range = (
            peak_nav - trough_nav
        )

        if total_range > 0:

            recovered = (
                (current_nav - trough_nav)
                / total_range
                * 100
            )

            recovered = np.clip(
                recovered,
                0,
                100
            )

        else:

            recovered = 0.0

    else:

        trough_nav = current_nav
        trough_date = current_date
        recovered = np.nan


    # --------------------------------------------------------
    # Cycle phase
    # --------------------------------------------------------

    if current_correction <= 0.5:

        cycle_phase = "AT / NEAR PEAK"

    elif current_nav <= trough_nav * 1.02:

        cycle_phase = "TOWARD TROUGHT"

    elif recovered >= 70:

        cycle_phase = "RECOVERING TOWARD PEAK"

    else:

        cycle_phase = "CORRECTING / MID-CYCLE"


    # --------------------------------------------------------
    # Historical correction distribution
    # --------------------------------------------------------

    corrections = (
        completed["Correction_%"]
        .dropna()
        if not completed.empty
        else pd.Series(dtype=float)
    )

    if len(corrections) > 0:

        p50 = corrections.quantile(0.50)
        p75 = corrections.quantile(0.75)
        p90 = corrections.quantile(0.90)
        p95 = corrections.quantile(0.95)
        max_corr = corrections.max()

    else:

        p50 = p75 = p90 = p95 = max_corr = np.nan


    return {

        "Current_Date":
            current_date,

        "Current_NAV":
            current_nav,

        "Peak_NAV":
            peak_nav,

        "Peak_Date":
            peak_date,

        "Current_Correction_%":
            current_correction,

        "Trough_NAV":
            trough_nav,

        "Trough_Date":
            trough_date,

        "Recovered_%":
            recovered,

        "Cycle_Phase":
            cycle_phase,

        "Median_Correction_%":
            p50,

        "P75_Correction_%":
            p75,

        "P90_Correction_%":
            p90,

        "P95_Correction_%":
            p95,

        "Max_Correction_%":
            max_corr,

        "Completed_Episodes":
            len(completed),

        "Open_Episodes":
            len(open_eps)
    }

In [19]:
# ============================================================
# SECTION 8 — BUILD diagnostics1_df
# ============================================================

def build_diagnostics1(
    df_nav,
    episodes_df
):

    rows = []

    for scheme_code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        fund_episodes = episodes_df[
            episodes_df["Scheme_Code"]
            == scheme_code
        ].copy()

        state = get_current_state(
            fund_df,
            fund_episodes
        )

        rows.append({

            "Scheme_Code":
                scheme_code,

            "Scheme_Name":
                fund_df["Scheme_Name"].iloc[0],

            **state

        })


    diagnostics1_df = pd.DataFrame(
        rows
    )

    return diagnostics1_df


diagnostics1_df = build_diagnostics1(
    df_nav,
    episodes_df
)

print(
    diagnostics1_df.shape
)

display(
    diagnostics1_df
)

(51, 18)


,Scheme_Code,Scheme_Name,Current_Date,Current_NAV,Peak_NAV,Peak_Date,Current_Correction_%,Trough_NAV,Trough_Date,Recovered_%,Cycle_Phase,Median_Correction_%,P75_Correction_%,P90_Correction_%,P95_Correction_%,Max_Correction_%,Completed_Episodes,Open_Episodes
0,103819,DSP Large & Mid Cap - Regular,2026-08-28,625.4150,650.6240,2026-01-06,3.874588,558.4130,2026-03-31,72.661613,RECOVERING TOWARD PEAK,5.640558,9.899674,19.619841,31.038473,61.833974,46,1
1,109445,ICICI Prudential Banking & Financial Services,2026-08-28,131.9200,140.8400,2026-01-06,6.333428,116.6600,2026-03-30,63.110008,CORRECTING / MID-CYCLE,4.972668,10.934984,19.988171,33.058359,47.848706,54,1
2,112932,Mirae Asset Large & Midcap - Regular,2026-08-28,159.7080,160.0690,2026-08-03,0.225528,157.9140,2026-08-19,83.248260,AT / NEAR PEAK,5.579188,10.152693,19.476001,20.405797,36.814192,36,0
3,118825,Mirae Asset Large Cap,2026-08-28,128.7350,134.2340,2026-01-02,4.096578,113.1210,2026-03-31,73.954436,RECOVERING TOWARD PEAK,5.719564,9.757778,15.599797,18.027102,37.441726,33,1
4,118834,Mirae Asset Large & Midcap - Direct,2026-08-28,181.5870,181.8840,2026-08-03,0.163291,179.5070,2026-08-19,87.505259,AT / NEAR PEAK,5.806890,9.067547,17.687313,19.095163,36.773423,35,0
5,118989,HDFC Mid Cap,2026-08-28,237.0660,237.6290,2026-08-25,0.236924,236.9190,2026-08-27,20.704225,AT / NEAR PEAK,5.885902,8.039246,16.916245,18.706732,39.513231,33,0
6,119063,HDFC Nifty 50,2026-08-28,236.6091,255.8052,2026-01-02,7.504187,217.1184,2026-03-31,50.380750,CORRECTING / MID-CYCLE,5.290830,9.880066,14.962378,17.835379,38.359143,36,1
7,119069,HDFC Income Fund - Growth Option - Direct Plan,2026-08-28,67.4577,68.0447,2026-08-14,0.862668,67.4577,2026-08-28,0.000000,TOWARD TROUGHT,5.956170,8.298174,9.703377,10.171777,10.640178,3,0
8,119151,ESSEL FLEXIBLE INCOME FUND-DIRECT PLAN-GROWTH ...,2018-08-20,14.9468,14.9468,2018-08-20,0.000000,14.9468,2018-08-20,NaN,AT / NEAR PEAK,3.001365,3.001365,3.001365,3.001365,3.001365,1,0
9,119218,DSP Large & Mid Cap - Direct,2026-08-28,706.8480,730.4570,2026-01-06,3.232086,628.4300,2026-03-31,76.860047,RECOVERING TOWARD PEAK,5.504114,8.165456,17.354251,19.109604,36.550601,35,1


In [20]:
# ============================================================
# SECTION 9 — DIAGNOSTICS 2
# HISTORICAL BUYING OPPORTUNITY ROBUSTNESS
# ============================================================

def classify_correction_signal(
    correction,
    p50,
    p75,
    p90,
    p95
):

    if pd.isna(correction):
        return "UNKNOWN"

    if pd.isna(p50):
        return "INSUFFICIENT HISTORY"

    if correction < p50:
        return "NORMAL"

    if correction < p75:
        return "WATCH"

    if correction < p90:
        return "ACCUMULATE"

    return "STRONG ACCUMULATE"


def build_diagnostics2(
    diagnostics1_df,
    episodes_df
):

    rows = []

    for _, current in diagnostics1_df.iterrows():

        code = int(
            current["Scheme_Code"]
        )

        current_corr = float(
            current["Current_Correction_%"]
        )

        historical = episodes_df[
            (
                episodes_df["Scheme_Code"]
                == code
            )
            &
            (
                episodes_df["Recovery_Date"]
                .notna()
            )
        ].copy()


        if historical.empty:

            rows.append({

                "Scheme_Code": code,

                "Scheme_Name":
                    current["Scheme_Name"],

                "Current_Correction_%":
                    current_corr,

                "Historical_Episodes":
                    0,

                "Deeper_Than_Today":
                    0,

                "Near_Today":
                    0,

                "Best_Historical_Correction_%":
                    np.nan,

                "Current_vs_P95":
                    np.nan,

                "Robustness":
                    "LOW"

            })

            continue


        corrections = historical[
            "Correction_%"
        ].dropna()


        p50 = corrections.quantile(
            0.50
        )

        p75 = corrections.quantile(
            0.75
        )

        p90 = corrections.quantile(
            0.90
        )

        p95 = corrections.quantile(
            0.95
        )


        historical[
            "Historical_Signal"
        ] = historical[
            "Correction_%"
        ].apply(
            lambda x:
                classify_correction_signal(
                    x,
                    p50,
                    p75,
                    p90,
                    p95
                )
        )


        deeper = historical[
            historical["Correction_%"]
            >= current_corr
        ]


        near = historical[
            (
                historical["Correction_%"]
                < current_corr
            )
            &
            (
                historical["Correction_%"]
                >=
                current_corr
                - NEAR_TODAY_TOLERANCE
            )
        ]


        rows.append({

            "Scheme_Code":
                code,

            "Scheme_Name":
                current["Scheme_Name"],

            "Current_Correction_%":
                current_corr,

            "P50":
                p50,

            "P75":
                p75,

            "P90":
                p90,

            "P95":
                p95,

            "Best_Historical_Correction_%":
                corrections.max(),

            "Historical_Episodes":
                len(historical),

            "Deeper_Than_Today":
                len(deeper),

            "Near_Today":
                len(near),

            "Current_Signal":
                classify_correction_signal(
                    current_corr,
                    p50,
                    p75,
                    p90,
                    p95
                ),

            "Robustness":
                (
                    "HIGH"
                    if len(historical) >= 20
                    else
                    "MEDIUM"
                    if len(historical) >= 10
                    else
                    "LOW"
                )

        })


    return pd.DataFrame(
        rows
    )


diagnostics2_df = build_diagnostics2(
    diagnostics1_df,
    episodes_df
)

display(
    diagnostics2_df
)

,Scheme_Code,Scheme_Name,Current_Correction_%,P50,P75,P90,P95,Best_Historical_Correction_%,Historical_Episodes,Deeper_Than_Today,Near_Today,Current_Signal,Robustness
0,103819,DSP Large & Mid Cap - Regular,3.874588,5.640558,9.899674,19.619841,31.038473,61.833974,46,36,10,NORMAL,HIGH
1,109445,ICICI Prudential Banking & Financial Services,6.333428,4.972668,10.934984,19.988171,33.058359,47.848706,54,17,7,WATCH,HIGH
2,112932,Mirae Asset Large & Midcap - Regular,0.225528,5.579188,10.152693,19.476001,20.405797,36.814192,36,36,0,NORMAL,HIGH
3,118825,Mirae Asset Large Cap,4.096578,5.719564,9.757778,15.599797,18.027102,37.441726,33,24,9,NORMAL,HIGH
4,118834,Mirae Asset Large & Midcap - Direct,0.163291,5.806890,9.067547,17.687313,19.095163,36.773423,35,35,0,NORMAL,HIGH
5,118989,HDFC Mid Cap,0.236924,5.885902,8.039246,16.916245,18.706732,39.513231,33,33,0,NORMAL,HIGH
6,119063,HDFC Nifty 50,7.504187,5.290830,9.880066,14.962378,17.835379,38.359143,36,11,2,WATCH,HIGH
7,119069,HDFC Income Fund - Growth Option - Direct Plan,0.862668,5.956170,8.298174,9.703377,10.171777,10.640178,3,3,0,NORMAL,LOW
8,119151,ESSEL FLEXIBLE INCOME FUND-DIRECT PLAN-GROWTH ...,0.000000,3.001365,3.001365,3.001365,3.001365,3.001365,1,1,0,NORMAL,LOW
9,119218,DSP Large & Mid Cap - Direct,3.232086,5.504114,8.165456,17.354251,19.109604,36.550601,35,31,4,NORMAL,HIGH


In [21]:
# ============================================================
# SECTION 10 — HISTORICAL OPPORTUNITY DETAIL
# ============================================================

def show_historical_opportunities(
    scheme_code,
    episodes_df,
    diagnostics2_df
):

    current = diagnostics2_df[
        diagnostics2_df["Scheme_Code"]
        == scheme_code
    ]

    if current.empty:

        print(
            f"No diagnostics found for {scheme_code}"
        )

        return


    current = current.iloc[0]

    historical = episodes_df[
        (
            episodes_df["Scheme_Code"]
            == scheme_code
        )
        &
        (
            episodes_df["Recovery_Date"]
            .notna()
        )
    ].copy()


    if historical.empty:

        print(
            "No completed historical episodes."
        )

        return


    current_corr = (
        current["Current_Correction_%"]
    )

    historical["Comparison"] = np.where(

        historical["Correction_%"]
        >= current_corr,

        "DEEPER_THAN_TODAY",

        np.where(

            historical["Correction_%"]
            >=
            current_corr
            - NEAR_TODAY_TOLERANCE,

            "NEAR_TODAY",

            "SHALLOWER"

        )
    )


    historical = historical[
        historical["Comparison"]
        .isin(
            [
                "DEEPER_THAN_TODAY",
                "NEAR_TODAY"
            ]
        )
    ]


    historical = historical.sort_values(
        "Correction_%",
        ascending=False
    )


    cols = [

        "Peak_Date",
        "Peak_NAV",

        "Trough_Date",
        "Trough_NAV",

        "Recovery_Date",

        "Correction_%",

        "Peak_to_Trough_Days",

        "Trough_to_Recovery_Days",

        "Total_Recovery_Days",

        "Comparison"

    ]


    print()
    print("=" * 110)
    print(
        current["Scheme_Name"]
    )
    print("=" * 110)

    print(
        f"Current correction: "
        f"{current_corr:.2f}%"
    )

    print()

    display(
        historical[cols]
        .reset_index(drop=True)
    )


# Example:
#
# show_historical_opportunities(
#     120594,
#     episodes_df,
#     diagnostics2_df
# )

In [22]:
# ============================================================
# SECTION 11 — PROFIT / OVERSHOOT ENGINE
# ============================================================

def build_profit_engine(
    df_nav
):

    rows = []

    for scheme_code, fund_df in df_nav.groupby(
        "Scheme_Code"
    ):

        fund_df = (
            fund_df
            .sort_values("Date")
            .reset_index(drop=True)
        )


        current_nav = float(
            fund_df["NAV"].iloc[-1]
        )

        current_date = (
            fund_df["Date"].iloc[-1]
        )


        # ----------------------------------------------------
        # Historical maximum before current date
        # ----------------------------------------------------

        peak_idx = fund_df["NAV"].idxmax()

        peak_nav = float(
            fund_df.loc[
                peak_idx,
                "NAV"
            ]
        )

        peak_date = pd.Timestamp(
            fund_df.loc[
                peak_idx,
                "Date"
            ]
        )


        overshoot_pct = (
            (
                current_nav
                /
                peak_nav
            )
            - 1
        ) * 100


        # ----------------------------------------------------
        # Profit-taking pressure
        # ----------------------------------------------------

        if overshoot_pct >= OVERSHOOT_PCT_3:

            trim_signal = (
                "STRONG TRIM"
            )

        elif overshoot_pct >= OVERSHOOT_PCT_2:

            trim_signal = (
                "TRIM"
            )

        elif overshoot_pct >= OVERSHOOT_PCT_1:

            trim_signal = (
                "WATCH TRIM"
            )

        else:

            trim_signal = (
                "NO TRIM"
            )


        rows.append({

            "Scheme_Code":
                scheme_code,

            "Scheme_Name":
                fund_df["Scheme_Name"].iloc[0],

            "Current_NAV":
                current_nav,

            "Current_Date":
                current_date,

            "Reference_Peak_NAV":
                peak_nav,

            "Reference_Peak_Date":
                peak_date,

            "Overshoot_%":
                overshoot_pct,

            "Profit_Taking_Signal":
                trim_signal

        })


    return pd.DataFrame(
        rows
    )


profit_df = build_profit_engine(
    df_nav
)

display(
    profit_df
)

,Scheme_Code,Scheme_Name,Current_NAV,Current_Date,Reference_Peak_NAV,Reference_Peak_Date,Overshoot_%,Profit_Taking_Signal
0,103819,DSP Large & Mid Cap - Regular,625.4150,2026-08-28,650.6240,2026-01-06,-3.874588,NO TRIM
1,109445,ICICI Prudential Banking & Financial Services,131.9200,2026-08-28,140.8400,2026-01-06,-6.333428,NO TRIM
2,112932,Mirae Asset Large & Midcap - Regular,159.7080,2026-08-28,160.0690,2026-08-03,-0.225528,NO TRIM
3,118825,Mirae Asset Large Cap,128.7350,2026-08-28,134.2340,2026-01-02,-4.096578,NO TRIM
4,118834,Mirae Asset Large & Midcap - Direct,181.5870,2026-08-28,181.8840,2026-08-03,-0.163291,NO TRIM
5,118989,HDFC Mid Cap,237.0660,2026-08-28,237.6290,2026-08-25,-0.236924,NO TRIM
6,119063,HDFC Nifty 50,236.6091,2026-08-28,255.8052,2026-01-02,-7.504187,NO TRIM
7,119069,HDFC Income Fund - Growth Option - Direct Plan,67.4577,2026-08-28,68.0447,2026-08-14,-0.862668,NO TRIM
8,119151,ESSEL FLEXIBLE INCOME FUND-DIRECT PLAN-GROWTH ...,14.9468,2018-08-20,14.9468,2018-08-20,0.000000,NO TRIM
9,119218,DSP Large & Mid Cap - Direct,706.8480,2026-08-28,730.4570,2026-01-06,-3.232086,NO TRIM


In [23]:
# ============================================================
# SECTION 12 — STRATEGIC GROUP / OVERLAP ENGINE
# ============================================================

def build_group_engine(
    diagnostics2_df,
    profit_df,
    strategic_groups
):

    rows = []


    for group_name, group_codes in strategic_groups.items():

        group_diag = diagnostics2_df[
            diagnostics2_df["Scheme_Code"]
            .isin(group_codes)
        ].copy()


        group_profit = profit_df[
            profit_df["Scheme_Code"]
            .isin(group_codes)
        ].copy()


        if group_diag.empty:
            continue


        # ----------------------------------------------------
        # Buy pressure
        # ----------------------------------------------------

        buy_scores = {

            "STRONG ACCUMULATE": 4,

            "ACCUMULATE": 3,

            "WATCH": 1,

            "NORMAL": 0

        }


        group_diag[
            "Buy_Score"
        ] = group_diag[
            "Current_Signal"
        ].map(
            buy_scores
        ).fillna(0)


        buy_pressure = (
            group_diag["Buy_Score"]
            .sum()
        )


        # ----------------------------------------------------
        # Maximum pressure
        # ----------------------------------------------------

        max_buy_score = (
            group_diag["Buy_Score"]
            .max()
        )


        # ----------------------------------------------------
        # Funds currently showing accumulation
        # ----------------------------------------------------

        accumulation_count = (
            group_diag[
                group_diag[
                    "Current_Signal"
                ].isin(
                    [
                        "ACCUMULATE",
                        "STRONG ACCUMULATE"
                    ]
                )
            ]
            .shape[0]
        )


        # ----------------------------------------------------
        # Profit-taking pressure
        # ----------------------------------------------------

        trim_map = {

            "STRONG TRIM": 4,

            "TRIM": 3,

            "WATCH TRIM": 1,

            "NO TRIM": 0

        }


        if not group_profit.empty:

            group_profit[
                "Trim_Score"
            ] = group_profit[
                "Profit_Taking_Signal"
            ].map(
                trim_map
            ).fillna(0)

            trim_pressure = (
                group_profit["Trim_Score"]
                .sum()
            )

        else:

            trim_pressure = 0


        # ----------------------------------------------------
        # Group conclusion
        # ----------------------------------------------------

        if (
            max_buy_score >= 4
            and accumulation_count >= 1
        ):

            group_signal = (
                "HIGH BUY PRESSURE"
            )

        elif accumulation_count >= 1:

            group_signal = (
                "BUY PRESSURE"
            )

        elif trim_pressure > 0:

            group_signal = (
                "TRIM PRESSURE"
            )

        else:

            group_signal = (
                "NEUTRAL"
            )


        rows.append({

            "Strategic_Group":
                group_name,

            "Funds":
                len(group_codes),

            "Funds_With_Data":
                len(group_diag),

            "Accumulation_Count":
                accumulation_count,

            "Buy_Pressure":
                buy_pressure,

            "Trim_Pressure":
                trim_pressure,

            "Group_Signal":
                group_signal

        })


    return pd.DataFrame(
        rows
    )


group_df = build_group_engine(
    diagnostics2_df,
    profit_df,
    strategic_groups
)

display(
    group_df
)

,Strategic_Group,Funds,Funds_With_Data,Accumulation_Count,Buy_Pressure,Trim_Pressure,Group_Signal
0,CORE_EQUITY,10,10,0,2,0,NEUTRAL
1,GROWTH_EQUITY,12,12,0,1,0,NEUTRAL
2,FACTOR_EQUITY,2,2,2,8,0,HIGH BUY PRESSURE
3,SECTOR_EQUITY,14,14,2,10,0,HIGH BUY PRESSURE
4,INTERNATIONAL,4,4,1,3,0,BUY PRESSURE
5,HYBRID_DEBT_OTHER,5,5,0,0,0,NEUTRAL


In [24]:
# ============================================================
# SECTION 13 — INVESTMENT ENGINE
# ============================================================

def build_investment_engine(
    diagnostics2_df,
    group_df,
    invested_codes,
    monthly_investment
):

    df = diagnostics2_df.copy()


    # --------------------------------------------------------
    # Base investment score
    # --------------------------------------------------------

    signal_score = {

        "STRONG ACCUMULATE": 4,

        "ACCUMULATE": 3,

        "WATCH": 1,

        "NORMAL": 0,

        "INSUFFICIENT HISTORY": 0

    }


    df["Signal_Score"] = (
        df["Current_Signal"]
        .map(signal_score)
        .fillna(0)
    )


    # --------------------------------------------------------
    # Historical robustness
    # --------------------------------------------------------

    robustness_score = {

        "HIGH": 2,

        "MEDIUM": 1,

        "LOW": 0

    }


    df["Robustness_Score"] = (
        df["Robustness"]
        .map(robustness_score)
        .fillna(0)
    )


    # --------------------------------------------------------
    # Deeper-than-today evidence
    # --------------------------------------------------------

    df["Depth_Score"] = np.select(

        [

            df["Deeper_Than_Today"] >= 3,

            df["Deeper_Than_Today"] >= 1

        ],

        [

            2,

            1

        ],

        default=0

    )


    # --------------------------------------------------------
    # Total score
    # --------------------------------------------------------

    df["Investment_Score"] = (

        df["Signal_Score"]

        + df["Robustness_Score"]

        + df["Depth_Score"]

    )


    # --------------------------------------------------------
    # Existing investment
    # --------------------------------------------------------

    df["Already_Invested"] = (
        df["Scheme_Code"]
        .isin(invested_codes)
    )


    # --------------------------------------------------------
    # Only funds with meaningful buy signals
    # --------------------------------------------------------

    candidates = df[
        df["Current_Signal"].isin(
            [
                "ACCUMULATE",
                "STRONG ACCUMULATE"
            ]
        )
    ].copy()


    if candidates.empty:

        df["Investment_₹"] = 0.0

        df["Investment_Action"] = (
            "NO NEW INVESTMENT"
        )

        return df


    # --------------------------------------------------------
    # Allocate according to score
    # --------------------------------------------------------

    candidates["Weight"] = (
        candidates["Investment_Score"]
        /
        candidates["Investment_Score"].sum()
    )


    candidates["Investment_₹"] = (
        candidates["Weight"]
        * monthly_investment
    )


    # --------------------------------------------------------
    # Round to practical amounts
    # --------------------------------------------------------

    candidates["Investment_₹"] = (
        candidates["Investment_₹"]
        .round(-3)
    )


    candidates["Investment_Action"] = (
        "INVEST"
    )


    # --------------------------------------------------------
    # Merge back
    # --------------------------------------------------------

    df["Investment_₹"] = 0.0

    df["Investment_Action"] = (
        "NO NEW INVESTMENT"
    )


    for _, row in candidates.iterrows():

        idx = (
            df["Scheme_Code"]
            == row["Scheme_Code"]
        )

        df.loc[
            idx,
            "Investment_₹"
        ] = row["Investment_₹"]

        df.loc[
            idx,
            "Investment_Action"
        ] = "INVEST"


    return df

In [25]:
# ============================================================
# SECTION 14 — BUILD INVESTMENT OUTPUT
# ============================================================

investment_df = build_investment_engine(
    diagnostics2_df,
    group_df,
    invested_codes,
    MONTHLY_INVESTMENT
)

investment_df = investment_df.sort_values(
    [
        "Investment_₹",
        "Investment_Score"
    ],
    ascending=False
)

display(
    investment_df[
        [
            "Scheme_Code",
            "Scheme_Name",
            "Current_Correction_%",
            "Current_Signal",
            "Robustness",
            "Historical_Episodes",
            "Deeper_Than_Today",
            "Investment_Score",
            "Already_Invested",
            "Investment_₹",
            "Investment_Action"
        ]
    ]
)

,Scheme_Code,Scheme_Name,Current_Correction_%,Current_Signal,Robustness,Historical_Episodes,Deeper_Than_Today,Investment_Score,Already_Invested,Investment_₹,Investment_Action
25,120594,ICICI Prudential Technology,16.336653,STRONG ACCUMULATE,HIGH,27,3,8,True,44000.0,INVEST
24,120578,SBI Technology Opportunities,11.486954,ACCUMULATE,HIGH,32,7,7,True,39000.0,INVEST
35,138528,PGIM India Global Equity Opportunities FoF,11.436718,ACCUMULATE,HIGH,37,6,7,True,39000.0,INVEST
46,148703,UTI Nifty 200 Momentum 30,16.757314,STRONG ACCUMULATE,MEDIUM,16,1,6,False,33000.0,INVEST
48,152417,NoName,7.199176,ACCUMULATE,LOW,6,1,4,False,22000.0,INVEST
49,152430,HDFC Nifty 200 Momentum 30,17.456870,STRONG ACCUMULATE,LOW,7,0,4,True,22000.0,INVEST
1,109445,ICICI Prudential Banking & Financial Services,6.333428,WATCH,HIGH,54,17,5,False,0.0,NO NEW INVESTMENT
6,119063,HDFC Nifty 50,7.504187,WATCH,HIGH,36,11,5,True,0.0,NO NEW INVESTMENT
18,120244,ICICI Prudential Banking & Financial Services ...,5.881244,WATCH,HIGH,39,14,5,True,0.0,NO NEW INVESTMENT
26,120620,ICICI Prudential Nifty 50,7.467437,WATCH,HIGH,36,11,5,False,0.0,NO NEW INVESTMENT


In [26]:
# ============================================================
# SECTION 15 — FINAL PORTFOLIO DECISION VIEW
# ============================================================

final_columns = [

    "Scheme_Code",
    "Scheme_Name",

    "Current_NAV",
    "Current_Correction_%",

    "P50",
    "P75",
    "P90",
    "P95",

    "Current_Signal",

    "Historical_Episodes",
    "Deeper_Than_Today",
    "Near_Today",

    "Robustness",

    "Already_Invested",

    "Investment_Score",
    "Investment_₹",
    "Investment_Action"

]


final_df = investment_df[
    [
        c
        for c in final_columns
        if c in investment_df.columns
    ]
].copy()


display(
    final_df
)

,Scheme_Code,Scheme_Name,Current_Correction_%,P50,P75,P90,P95,Current_Signal,Historical_Episodes,Deeper_Than_Today,Near_Today,Robustness,Already_Invested,Investment_Score,Investment_₹,Investment_Action
25,120594,ICICI Prudential Technology,16.336653,5.865892,12.581792,16.323011,25.481032,STRONG ACCUMULATE,27,3,2,HIGH,True,8,44000.0,INVEST
24,120578,SBI Technology Opportunities,11.486954,5.329085,9.316319,17.420976,23.408808,ACCUMULATE,32,7,0,HIGH,True,7,39000.0,INVEST
35,138528,PGIM India Global Equity Opportunities FoF,11.436718,5.323958,8.145106,18.516802,26.208018,ACCUMULATE,37,6,1,HIGH,True,7,39000.0,INVEST
46,148703,UTI Nifty 200 Momentum 30,16.757314,3.878285,6.310544,8.537584,14.313677,STRONG ACCUMULATE,16,1,0,MEDIUM,False,6,33000.0,INVEST
48,152417,NoName,7.199176,5.120742,5.441170,7.957241,9.209406,ACCUMULATE,6,1,0,LOW,False,4,22000.0,INVEST
49,152430,HDFC Nifty 200 Momentum 30,17.456870,5.028115,7.230951,8.285459,9.033103,STRONG ACCUMULATE,7,0,0,LOW,True,4,22000.0,INVEST
1,109445,ICICI Prudential Banking & Financial Services,6.333428,4.972668,10.934984,19.988171,33.058359,WATCH,54,17,7,HIGH,False,5,0.0,NO NEW INVESTMENT
6,119063,HDFC Nifty 50,7.504187,5.290830,9.880066,14.962378,17.835379,WATCH,36,11,2,HIGH,True,5,0.0,NO NEW INVESTMENT
18,120244,ICICI Prudential Banking & Financial Services ...,5.881244,4.709511,11.083004,18.576140,27.776419,WATCH,39,14,5,HIGH,True,5,0.0,NO NEW INVESTMENT
26,120620,ICICI Prudential Nifty 50,7.467437,5.363596,9.913461,14.982868,17.849149,WATCH,36,11,2,HIGH,False,5,0.0,NO NEW INVESTMENT


In [28]:
# ============================================================
# SECTION 16 — SINGLE FUND DEEP DIAGNOSTIC
# ============================================================

def deep_diagnostic(
    scheme_code
):

    name = FUND_NAMES.get(
        scheme_code,
        "NoName"
    )

    print()
    print("=" * 110)
    print(
        f"DEEP DIAGNOSTIC — {name}"
    )
    print(
        f"Scheme Code: {scheme_code}"
    )
    print("=" * 110)


    # --------------------------------------------------------
    # Diagnostics 1
    # --------------------------------------------------------

    d1 = diagnostics1_df[
        diagnostics1_df["Scheme_Code"]
        == scheme_code
    ]

    print()
    print("CURRENT STATE")
    print("-" * 110)

    display(d1)


    # --------------------------------------------------------
    # Diagnostics 2
    # --------------------------------------------------------

    d2 = diagnostics2_df[
        diagnostics2_df["Scheme_Code"]
        == scheme_code
    ]

    print()
    print("HISTORICAL ROBUSTNESS")
    print("-" * 110)

    display(d2)


    # --------------------------------------------------------
    # Profit engine
    # --------------------------------------------------------

    p = profit_df[
        profit_df["Scheme_Code"]
        == scheme_code
    ]

    print()
    print("PROFIT / TRIM ENGINE")
    print("-" * 110)

    display(p)


    # --------------------------------------------------------
    # Historical episodes
    # --------------------------------------------------------

    print()
    print("CORRECTION EPISODES")
    print("-" * 110)

    ep = episodes_df[
        episodes_df["Scheme_Code"]
        == scheme_code
    ].copy()

    display(ep)


    # --------------------------------------------------------
    # Historical opportunities
    # --------------------------------------------------------

    show_historical_opportunities(
        scheme_code,
        episodes_df,
        diagnostics2_df
    )


# Example:
# deep_diagnostic(127042)
# deep_diagnostic(120594)
# deep_diagnostic(152430)
deep_diagnostic(138528)


DEEP DIAGNOSTIC — PGIM India Global Equity Opportunities FoF
Scheme Code: 138528

CURRENT STATE
--------------------------------------------------------------------------------------------------------------


,Scheme_Code,Scheme_Name,Current_Date,Current_NAV,Peak_NAV,Peak_Date,Current_Correction_%,Trough_NAV,Trough_Date,Recovered_%,Cycle_Phase,Median_Correction_%,P75_Correction_%,P90_Correction_%,P95_Correction_%,Max_Correction_%,Completed_Episodes,Open_Episodes
35,138528,PGIM India Global Equity Opportunities FoF,2026-08-28,55.91,63.13,2026-06-22,11.436718,53.48,2026-07-29,25.181347,CORRECTING / MID-CYCLE,5.323958,8.145106,18.516802,26.208018,43.424263,37,1



HISTORICAL ROBUSTNESS
--------------------------------------------------------------------------------------------------------------


,Scheme_Code,Scheme_Name,Current_Correction_%,P50,P75,P90,P95,Best_Historical_Correction_%,Historical_Episodes,Deeper_Than_Today,Near_Today,Current_Signal,Robustness
35,138528,PGIM India Global Equity Opportunities FoF,11.436718,5.323958,8.145106,18.516802,26.208018,43.424263,37,6,1,ACCUMULATE,HIGH



PROFIT / TRIM ENGINE
--------------------------------------------------------------------------------------------------------------


,Scheme_Code,Scheme_Name,Current_NAV,Current_Date,Reference_Peak_NAV,Reference_Peak_Date,Overshoot_%,Profit_Taking_Signal
35,138528,PGIM India Global Equity Opportunities FoF,55.91,2026-08-28,63.13,2026-06-22,-11.436718,NO TRIM



CORRECTION EPISODES
--------------------------------------------------------------------------------------------------------------


,Scheme_Code,Scheme_Name,Peak_Date,Peak_NAV,Trough_Date,Trough_NAV,Recovery_Date,Recovery_NAV,Correction_%,Peak_to_Trough_Days,Trough_to_Recovery_Days,Total_Recovery_Days,Recovered_%
1100,138528,PGIM India Global Equity Opportunities FoF,2016-03-18,13.97,2016-04-06,13.25,2016-04-20,14.02,5.153901,19,14.0,33.0,100.000000
1101,138528,PGIM India Global Equity Opportunities FoF,2016-04-21,14.08,2016-05-09,13.42,2016-05-26,14.03,4.687500,18,17.0,35.0,100.000000
1102,138528,PGIM India Global Equity Opportunities FoF,2016-06-08,14.61,2016-06-27,13.65,2016-08-23,14.54,6.570842,19,57.0,76.0,100.000000
1103,138528,PGIM India Global Equity Opportunities FoF,2016-08-31,14.61,2016-11-04,13.42,2017-01-10,14.54,8.145106,65,67.0,132.0,100.000000
1104,138528,PGIM India Global Equity Opportunities FoF,2017-01-25,15.14,2017-04-21,13.93,2017-09-11,15.09,7.992074,86,143.0,229.0,100.000000
1105,138528,PGIM India Global Equity Opportunities FoF,2018-01-24,16.81,2018-04-04,15.61,2018-05-14,16.83,7.138608,70,40.0,110.0,100.000000
1106,138528,PGIM India Global Equity Opportunities FoF,2018-09-25,19.12,2018-12-28,15.68,2019-02-06,19.14,17.991632,94,40.0,134.0,100.000000
1107,138528,PGIM India Global Equity Opportunities FoF,2019-02-28,19.51,2019-03-11,18.83,2019-03-22,19.42,3.485392,11,11.0,22.0,100.000000
1108,138528,PGIM India Global Equity Opportunities FoF,2019-04-30,20.01,2019-05-14,19.27,2019-05-17,19.91,3.698151,14,3.0,17.0,100.000000
1109,138528,PGIM India Global Equity Opportunities FoF,2019-05-17,19.91,2019-06-04,18.85,2019-06-11,19.85,5.323958,18,7.0,25.0,100.000000



PGIM India Global Equity Opportunities FoF
Current correction: 11.44%



,Peak_Date,Peak_NAV,Trough_Date,Trough_NAV,Recovery_Date,Correction_%,Peak_to_Trough_Days,Trough_to_Recovery_Days,Total_Recovery_Days,Comparison
0,2021-11-08,45.09,2022-06-16,25.51,2024-03-07,43.424263,220,630.0,850.0,DEEPER_THAN_TODAY
1,2020-02-20,24.55,2020-03-18,17.98,2020-05-11,26.761711,27,54.0,81.0,DEEPER_THAN_TODAY
2,2025-02-10,52.59,2025-04-07,38.88,2025-09-08,26.069595,56,154.0,210.0,DEEPER_THAN_TODAY
3,2021-02-16,41.70,2021-05-14,33.65,2021-08-05,19.304556,87,83.0,170.0,DEEPER_THAN_TODAY
4,2018-09-25,19.12,2018-12-28,15.68,2019-02-06,17.991632,94,40.0,134.0,DEEPER_THAN_TODAY
5,2025-10-29,55.67,2026-03-30,47.28,2026-04-22,15.070954,152,23.0,175.0,DEEPER_THAN_TODAY
6,2024-07-10,48.55,2024-08-07,43.11,2024-10-09,11.204943,28,63.0,91.0,NEAR_TODAY
